# Organizador final - Importacao, recorrencia e entrega

Execute em ordem. Este notebook e independente dos arquivos Python do computador. Nao requer instalacao de bibliotecas nem chave de API. Depois de editar uma definicao, execute sua celula novamente. Salve ou baixe sua copia.


## Definicoes


In [ ]:
"""Regras da referência final de estudo. Python 3.12, sem rede ou bibliotecas externas."""

from copy import deepcopy
from datetime import date, timedelta

ERRO_TITULO = "Informe um título com pelo menos um caractere que não seja espaço."
ERRO_PRIORIDADE = "Prioridade inválida. Use baixa, media ou alta."
PRIORIDADES = ("baixa", "media", "alta")


def interpretar_prazo(texto):
    if texto is None:
        return None
    if not isinstance(texto, str):
        raise ValueError("Prazo deve ser texto no formato AAAA-MM-DD.")
    try:
        prazo = date.fromisoformat(texto)
    except ValueError:
        raise ValueError("Prazo inválido. Use AAAA-MM-DD.") from None
    if prazo.isoformat() != texto:
        raise ValueError("Prazo inválido. Use AAAA-MM-DD.")
    return prazo


def cadastrar_tarefa(tarefas, titulo, prioridade="media", prazo=None, intervalo_dias=None):
    if not isinstance(titulo, str) or not titulo.strip():
        return ERRO_TITULO
    if prioridade not in PRIORIDADES:
        return ERRO_PRIORIDADE
    try:
        data_prazo = interpretar_prazo(prazo)
    except ValueError as erro:
        return str(erro)
    if intervalo_dias is not None:
        if type(intervalo_dias) is not int or not 1 <= intervalo_dias <= 999999999:
            return "Intervalo deve ser um inteiro positivo de até 999999999 dias."
        if data_prazo is None:
            return "Tarefa recorrente precisa de prazo."
    tarefa = {
        "titulo": titulo,
        "concluida": False,
        "prioridade": prioridade,
        "prazo": data_prazo,
    }
    if intervalo_dias is not None:
        tarefa["intervalo_dias"] = intervalo_dias
    tarefas.append(tarefa)
    return "Tarefa cadastrada."


def concluir_tarefa(tarefas, indice):
    if type(indice) is not int or indice < 0 or indice >= len(tarefas):
        return "Tarefa não encontrada."
    tarefa = tarefas[indice]
    if tarefa["concluida"]:
        return "Tarefa concluída."
    intervalo = tarefa.get("intervalo_dias")
    proxima = None
    if intervalo is not None:
        try:
            novo_prazo = tarefa["prazo"] + timedelta(days=intervalo)
        except (OverflowError, TypeError):
            return "Próximo prazo fora do intervalo suportado."
        proxima = {
            "titulo": tarefa["titulo"],
            "concluida": False,
            "prioridade": tarefa.get("prioridade", "media"),
            "prazo": novo_prazo,
            "intervalo_dias": intervalo,
        }
    tarefa["concluida"] = True
    if proxima is not None:
        tarefa["proxima_gerada"] = True
        tarefas.append(proxima)
    return "Tarefa concluída."


def filtrar_tarefas(tarefas, prioridade=None, concluida=None):
    if prioridade is not None and prioridade not in PRIORIDADES:
        raise ValueError("Prioridade de filtro inválida.")
    if concluida is not None and type(concluida) is not bool:
        raise ValueError("Conclusão de filtro deve ser booleana.")
    resultado = []
    for tarefa in tarefas:
        if prioridade is not None and tarefa.get("prioridade", "media") != prioridade:
            continue
        if concluida is not None and tarefa["concluida"] != concluida:
            continue
        resultado.append(deepcopy(tarefa))
    return resultado


def esta_atrasada(tarefa, hoje):
    prazo = tarefa.get("prazo")
    return (
        not tarefa["concluida"]
        and prazo is not None
        and prazo < hoje
    )

"""Apresentação textual preservada desde o primeiro exemplo."""


def listar_tarefas(tarefas):
    if not tarefas:
        return "Nenhuma tarefa cadastrada."
    linhas = []
    for indice, tarefa in enumerate(tarefas):
        marca = "x" if tarefa["concluida"] else " "
        linhas.append(f"{indice}: [{marca}] {tarefa['titulo']}")
    return "\n".join(linhas)

"""Importação/exportação de texto JSON; não grava arquivos por conta própria."""

import json


LIMITE_BYTES = 256000
LIMITE_REGISTROS = 1000
CAMPOS = {"titulo", "concluida", "prioridade", "prazo", "intervalo_dias", "proxima_gerada"}


def objeto_sem_chaves_duplicadas(pares):
    objeto = {}
    for chave, valor in pares:
        if chave in objeto:
            raise ValueError(f"Chave duplicada: {chave}.")
        objeto[chave] = valor
    return objeto


def validar_registro(registro):
    if not isinstance(registro, dict):
        raise ValueError("Cada tarefa deve ser um objeto.")
    if set(registro) - CAMPOS:
        raise ValueError("Campos desconhecidos.")
    if "titulo" not in registro or "concluida" not in registro:
        raise ValueError("Título e conclusão são obrigatórios.")
    if type(registro["concluida"]) is not bool:
        raise ValueError("Conclusão deve ser booleana.")
    if "proxima_gerada" in registro and type(registro["proxima_gerada"]) is not bool:
        raise ValueError("Marcador de próxima ocorrência deve ser booleano.")
    temporarias = []
    mensagem = cadastrar_tarefa(
        temporarias,
        registro["titulo"],
        prioridade=registro.get("prioridade", "media"),
        prazo=registro.get("prazo"),
        intervalo_dias=registro.get("intervalo_dias"),
    )
    if mensagem != "Tarefa cadastrada.":
        raise ValueError(mensagem)
    tarefa = temporarias[0]
    tarefa["concluida"] = registro["concluida"]
    if "proxima_gerada" in registro:
        if "intervalo_dias" not in tarefa:
            raise ValueError("Marcador de recorrência em tarefa comum.")
        if registro["proxima_gerada"] and not tarefa["concluida"]:
            raise ValueError("Ocorrência aberta não pode ter gerado a próxima.")
        tarefa["proxima_gerada"] = registro["proxima_gerada"]
    return tarefa


def importar_json(tarefas, texto):
    if not isinstance(texto, str):
        return "Importação recusada: forneça texto JSON."
    try:
        tamanho = len(texto.encode("utf-8"))
    except UnicodeError:
        return "Importação recusada: texto Unicode inválido."
    if tamanho > LIMITE_BYTES:
        return "Importação recusada: limite de tamanho excedido."
    try:
        registros = json.loads(texto, object_pairs_hook=objeto_sem_chaves_duplicadas)
    except (ValueError, RecursionError):
        return "Importação recusada: JSON inválido ou chave duplicada."
    if not isinstance(registros, list) or len(registros) > LIMITE_REGISTROS:
        return "Importação recusada: use uma lista com até 1000 registros."
    temporarias = []
    for indice, registro in enumerate(registros):
        try:
            temporarias.append(validar_registro(registro))
        except ValueError as erro:
            return f"Importação recusada no registro {indice}: {erro}"
    tarefas.extend(temporarias)
    return f"Importação concluída: {len(temporarias)} tarefa(s)."


def exportar_json(tarefas):
    registros = []
    for tarefa in tarefas:
        registro = dict(tarefa)
        prazo = registro.get("prazo")
        if prazo is not None:
            registro["prazo"] = prazo.isoformat()
        registros.append(registro)
    return json.dumps(registros, ensure_ascii=False, indent=2, allow_nan=False)


## Demonstracao


In [ ]:
"""Percurso completo de referência: não acessa rede nem grava arquivos."""

from datetime import date



def demonstrar():
    tarefas = []
    print(cadastrar_tarefa(tarefas, "Ler capítulo", prioridade="baixa"))
    print(cadastrar_tarefa(tarefas, "Revisar projeto", prioridade="alta",
                          prazo="2026-10-14", intervalo_dias=7))
    print("Abertas de prioridade alta:", len(filtrar_tarefas(tarefas, "alta", False)))
    texto = exportar_json(tarefas)
    recuperadas = []
    print(importar_json(recuperadas, texto))
    print(concluir_tarefa(recuperadas, 1))
    print(concluir_tarefa(recuperadas, 1))
    print(listar_tarefas(recuperadas))
    print("Próximo prazo:", recuperadas[2]["prazo"].isoformat())
    print("Primeira ocorrência atrasada após concluir:",
          esta_atrasada(recuperadas[1], date(2026, 10, 17)))
    return recuperadas



recuperadas = demonstrar()


## Conferencia dos casos

Uma falha pode produzir AssertionError. O resultado verifica estes casos, sem prometer ausencia de outros defeitos.


In [ ]:
assert len(recuperadas) == 3
assert recuperadas[2]["prazo"] == date(2026, 10, 21)
assert recuperadas[1]["concluida"] is True
antes = deepcopy(recuperadas)
texto_invalido = '[{"titulo":"Valida","concluida":false},{"titulo":"Invalida","concluida":false,"prioridade":"urgente"}]'
assert importar_json(recuperadas, texto_invalido).startswith("Importação recusada")
assert recuperadas == antes
copia = []
assert importar_json(copia, exportar_json(recuperadas)).startswith("Importação concluída")
assert copia == recuperadas
assert concluir_tarefa(copia, 1) == "Tarefa concluída."
assert len(copia) == 3
print("Integracao, importacao integral e recorrencia conferidas.")
